# Problem: Implement Multi-Head Attention from Scratch

### Problem Statement
You are tasked with implementing **Multi-Head Attention** mechanism from scratch using PyTorch. This is a core component of the Transformer architecture used in modern NLP models like BERT and GPT.

### Requirements
1. **Implement Custom Multi-Head Attention**:
   - Create a `MultiHeadAttention` class that implements the multi-head attention mechanism
   - Use linear projections for Query (Q), Key (K), and Value (V)
   - Implement scaled dot-product attention
   - Concatenate outputs from multiple heads and apply final linear projection

2. **Test the Implementation**:
   - Create random input tensors to simulate sequences
   - Verify output shapes are correct
   - Test with different configurations (sequence length, embedding dimension, number of heads)

### Steps

**Step 1: Import Libraries**
- Import PyTorch and necessary modules

**Step 2: Implement Scaled Dot-Product Attention**
- Create a function to compute attention scores
- Apply softmax and compute weighted values

**Step 3: Implement Multi-Head Attention**
- Create linear projections for Q, K, V
- Split into multiple heads
- Apply attention mechanism
- Concatenate and project outputs

**Step 4: Test with Random Tensors**
- Create random input tensors
- Verify output shapes
- Test with different parameters

### Constraints
- You must not use PyTorch's built-in `nn.MultiheadAttention`
- Implement the attention mechanism from first principles
- Ensure the implementation is batched (handles multiple sequences simultaneously)

<details>
  <summary>💡 Hint</summary>
  The attention formula is: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V
  
  For multi-head attention:
  1. Project input to Q, K, V using linear layers
  2. Split into num_heads (reshape and transpose)
  3. Apply scaled dot-product attention for each head
  4. Concatenate heads and apply final linear projection
</details>

In [ ]:
# Step 1: Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# Step 2: Implement Scaled Dot-Product Attention
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Query tensor of shape (batch_size, num_heads, seq_len, d_k)
        K: Key tensor of shape (batch_size, num_heads, seq_len, d_k)
        V: Value tensor of shape (batch_size, num_heads, seq_len, d_k)
        mask: Optional mask tensor
    
    Returns:
        output: Attention output of shape (batch_size, num_heads, seq_len, d_k)
        attention_weights: Attention weights of shape (batch_size, num_heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # Compute attention scores: Q @ K^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply mask if provided (set masked positions to large negative value)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Multiply by values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

In [ ]:
# Step 3: Implement Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        Initialize Multi-Head Attention.
        
        Args:
            d_model: Dimension of the model (embedding dimension)
            num_heads: Number of attention heads
        """
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Final output projection
        self.W_o = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, d_k).
        Transpose to shape: (batch_size, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, d_model = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine heads back to original shape.
        Input shape: (batch_size, num_heads, seq_len, d_k)
        Output shape: (batch_size, seq_len, d_model)
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)
    
    def forward(self, Q, K, V, mask=None):
        """
        Forward pass of Multi-Head Attention.
        
        Args:
            Q: Query tensor of shape (batch_size, seq_len, d_model)
            K: Key tensor of shape (batch_size, seq_len, d_model)
            V: Value tensor of shape (batch_size, seq_len, d_model)
            mask: Optional mask tensor
        
        Returns:
            output: Multi-head attention output of shape (batch_size, seq_len, d_model)
        """
        batch_size = Q.size(0)
        
        # Apply linear projections
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # Split into multiple heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Apply scaled dot-product attention
        attn_output, self.attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads
        attn_output = self.combine_heads(attn_output)
        
        # Apply final linear projection
        output = self.W_o(attn_output)
        
        return output

In [ ]:
# Step 4: Test with Random Tensors
torch.manual_seed(42)

# Define test parameters
batch_size = 2
seq_len = 10
d_model = 512
num_heads = 8

# Create random input tensors
Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

print("Input shapes:")
print(f"Q: {Q.shape}")
print(f"K: {K.shape}")
print(f"V: {V.shape}")

# Initialize MultiHeadAttention module
mha = MultiHeadAttention(d_model, num_heads)

# Forward pass
output = mha(Q, K, V)

# Print output shape
print(f"\nOutput shape: {output.shape}")
print(f"Expected shape: torch.Size([{batch_size}, {seq_len}, {d_model}])")

# Verify output shape
assert output.shape == (batch_size, seq_len, d_model), "Output shape mismatch!"
print("\n✓ Output shape verification passed!")

In [ ]:
# Additional Test: Different Configurations
test_configs = [
    {"batch_size": 4, "seq_len": 20, "d_model": 256, "num_heads": 4},
    {"batch_size": 1, "seq_len": 50, "d_model": 768, "num_heads": 12},
    {"batch_size": 8, "seq_len": 5, "d_model": 128, "num_heads": 8},
]

print("Testing different configurations:\n")
for i, config in enumerate(test_configs, 1):
    bs = config["batch_size"]
    sl = config["seq_len"]
    dm = config["d_model"]
    nh = config["num_heads"]
    
    # Create random tensors
    Q_test = torch.randn(bs, sl, dm)
    K_test = torch.randn(bs, sl, dm)
    V_test = torch.randn(bs, sl, dm)
    
    # Initialize and test
    mha_test = MultiHeadAttention(dm, nh)
    output_test = mha_test(Q_test, K_test, V_test)
    
    print(f"Config {i}:")
    print(f"  Parameters: batch_size={bs}, seq_len={sl}, d_model={dm}, num_heads={nh}")
    print(f"  Input shape: {Q_test.shape}")
    print(f"  Output shape: {output_test.shape}")
    assert output_test.shape == (bs, sl, dm), f"Config {i} failed!"
    print(f"  ✓ Passed\n")

print("All configuration tests passed!")

In [ ]:
# Verification: Check attention weights properties
print("Verifying attention weights properties:\n")

# Create test input
Q_verify = torch.randn(2, 10, 512)
K_verify = torch.randn(2, 10, 512)
V_verify = torch.randn(2, 10, 512)

mha_verify = MultiHeadAttention(512, 8)
output_verify = mha_verify(Q_verify, K_verify, V_verify)

# Check attention weights sum to 1
attn_weights = mha_verify.attention_weights
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Expected: (batch_size=2, num_heads=8, seq_len=10, seq_len=10)")

# Verify sum of attention weights along last dimension equals 1
attn_sum = attn_weights.sum(dim=-1)
print(f"\nSum of attention weights (should be all 1s):")
print(f"Min: {attn_sum.min().item():.6f}")
print(f"Max: {attn_sum.max().item():.6f}")
print(f"Mean: {attn_sum.mean().item():.6f}")

assert torch.allclose(attn_sum, torch.ones_like(attn_sum), atol=1e-6), "Attention weights don't sum to 1!"
print("\n✓ Attention weights sum to 1 (verified)")

# Check gradient flow
Q_grad = torch.randn(1, 5, 64, requires_grad=True)
K_grad = torch.randn(1, 5, 64, requires_grad=True)
V_grad = torch.randn(1, 5, 64, requires_grad=True)

mha_grad = MultiHeadAttention(64, 4)
output_grad = mha_grad(Q_grad, K_grad, V_grad)
loss = output_grad.sum()
loss.backward()

print(f"\nGradient flow check:")
print(f"Q gradient exists: {Q_grad.grad is not None}")
print(f"K gradient exists: {K_grad.grad is not None}")
print(f"V gradient exists: {V_grad.grad is not None}")
print("\n✓ Gradient flow verified")

print("\n" + "="*50)
print("All verifications passed! Multi-Head Attention implementation is correct.")
print("="*50)